# 🌊 SLCCI Full Analysis Pipeline
## Complete SLCCI along-track altimetry analysis — all dashboard tabs in one notebook

This notebook replicates **every tab** of the SLCCI Streamlit dashboard using `src/slcci` and `src/physics` modules:

| # | Section | Dashboard Tab |
|---|---------|---------------|
| 1 | Load & DOT | — |
| 2 | Slope Timeline | 📈 Slope Timeline |
| 3 | DOT Profile | 📊 DOT Profile |
| 4 | Spatial Map | 🗺️ Spatial Map |
| 5 | Monthly Analysis | 📅 Monthly Analysis |
| 6 | Geostrophic Velocity | 🌀 Geostrophic Velocity |
| 7 | Volume Transport | 🌊 Volume Transport |
| 8 | Freshwater Transport | 💧 Freshwater Transport |
| 9 | Salinity Profile | 🧂 Salinity Profile |
| 10 | Salt Flux | 🧪 Salt Flux |
| 11 | Temporal Aggregation | 📤 Export |

**Method**: SLCCI DOT = corrected SSH − geoid → linear slope → v_geo = −(g/f) × ∂η/∂x

In [1]:
# ════════════════════════════════════════════════════════════════
# 📚 CELL 1: IMPORTS
# ════════════════════════════════════════════════════════════════

import sys
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from pathlib import Path
from scipy import stats
from scipy.interpolate import RegularGridInterpolator
from shapely.geometry import LineString

# ── Add project root to path ──
PROJECT_ROOT = Path("__file__").resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# ── src/slcci — pure I/O & DOT ──
from src.slcci.loader import load_cycles_serial
from src.slcci.geoid import interpolate_geoid, load_geoid
from src.slcci.dot import compute_dot, build_dataframe, build_dot_matrix, compute_slope_series
from src.slcci.spatial import gate_bounds, gate_profile_points, filter_near_gate
from src.slcci.binning import longitude_bin, mean_profile_pooled, monthly_climatology_profiles

# ── src/physics — geostrophy, transport, aggregation ──
from src.physics.constants import GRAVITY, DEPTH_CAP, S_REF, RHO, SVERDRUP, coriolis_parameter
from src.physics.geostrophy import geostrophic_velocity_from_slope
from src.physics.transport import volume_transport, freshwater_transport, salt_flux
from src.physics.aggregation import monthly_mean, monthly_climatology, annual_mean, rolling_mean

# ── Matplotlib defaults ──
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": ":",
})

print("✅ All imports OK")

✅ All imports OK


In [2]:
# ════════════════════════════════════════════════════════════════
# ⚙️ CELL 2: CONFIGURATION
# ════════════════════════════════════════════════════════════════

# ── 1. STRAIT / PASS ──
SELECTED_STRAIT = "davis"          # "davis", "fram", "bering"
PASS_NUMBER     = 248

# ── 2. DATA PATHS ──
BASE_DIR   = Path("/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/J2")
GEOID_PATH = "/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/TUM_ogmoc.nc"
GEBCO_PATH = "/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/GEBCO_06_Feb_2026_c91df93f54b8/gebco_2025_n90.0_s55.0_w0.0_e360.0.nc"
PSAL_DIR   = Path("/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/ISAS25_PSAL")
GATES_FOLDER = str(PROJECT_ROOT / "gates")

# ── 3. PROCESSING ──
USE_FLAG      = True
LON_BIN_SIZE  = 0.01      # fine binning for slope/matrix
PROFILE_BIN   = 0.10      # coarser binning for profiles
DEPTH_CAP_M   = 250.0     # max integration depth [m]
S_REF_PSU     = 34.8      # reference salinity [PSU]

# ── 4. OUTPUT ──
SAVE_OUTPUTS = True
OUTPUT_DIR   = Path("/Users/nicolocaron/Desktop/ARCFRESH/plot slccii/FULL_ANALYSIS")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Satellite auto-detect ──
mission = BASE_DIR.name.upper()
if mission == "J2":
    cycles = list(range(1, 282))
    satellite_name = "J2"
elif mission == "J1":
    cycles = list(range(1, 538))
    satellite_name = "J1"
else:
    cycles = list(range(1, 282))
    satellite_name = mission

# ── Strait → file pattern map ──
STRAIT_PATTERNS = {
    "davis": "*davis_strait*.shp",
    "fram": "*fram*.shp",
    "bering": "*bering*.shp",
}

# ── Strait → salinity file map ──
STRAIT_PSAL_MAP = {
    "davis": "davis_strait_CLIM_ISAS25_PSAL.nc",
    "fram": "fram_strait_S3_pass_481_CLIM_ISAS25_PSAL.nc",
    "bering": "bering_strait_TPJ_pass_076_CLIM_ISAS25_PSAL.nc",
}

print("\n" + "=" * 70)
print("⚙️  CONFIGURATION")
print("=" * 70)
print(f"🌊 Strait:       {SELECTED_STRAIT.upper()}")
print(f"🛰️  Satellite:    {satellite_name} ({len(cycles)} cycles)")
print(f"🔢 Pass:         {PASS_NUMBER}")
print(f"📂 Data dir:     {BASE_DIR}")
print(f"📂 Gates:        {GATES_FOLDER}")
print(f"🌍 GEBCO:        {Path(GEBCO_PATH).name}")
print(f"🧂 Salinity:     {STRAIT_PSAL_MAP.get(SELECTED_STRAIT, 'N/A')}")
print(f"📊 Bin sizes:    slope={LON_BIN_SIZE}°, profile={PROFILE_BIN}°")
print(f"🏊 Depth cap:    {DEPTH_CAP_M} m")
print(f"💾 Output:       {OUTPUT_DIR}")
print("=" * 70)


⚙️  CONFIGURATION
🌊 Strait:       DAVIS
🛰️  Satellite:    J2 (281 cycles)
🔢 Pass:         248
📂 Data dir:     /Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/J2
📂 Gates:        /Users/nicolocaron/Documents/GitHub/ARCFRESH-DTU-NICO-and-AMALIE/gates
🌍 GEBCO:        gebco_2025_n90.0_s55.0_w0.0_e360.0.nc
🧂 Salinity:     davis_strait_CLIM_ISAS25_PSAL.nc
📊 Bin sizes:    slope=0.01°, profile=0.1°
🏊 Depth cap:    250.0 m
💾 Output:       /Users/nicolocaron/Desktop/ARCFRESH/plot slccii/FULL_ANALYSIS


In [3]:
# ════════════════════════════════════════════════════════════════
# 📡 CELL 3: LOAD DATA (Cycles → Geoid → DOT DataFrame)
# ════════════════════════════════════════════════════════════════

# ── 3a. Load gate shapefile ──
pattern = str(Path(GATES_FOLDER) / STRAIT_PATTERNS[SELECTED_STRAIT.lower()])
gate_files = [f for f in glob.glob(pattern) if "_east" not in f and "_west" not in f]
if not gate_files:
    raise FileNotFoundError(f"No gate file found for '{SELECTED_STRAIT}' with pattern {pattern}")
GATE_PATH = gate_files[0]

strait_name = Path(GATE_PATH).stem.replace("_", " ").title()
gate = gpd.read_file(GATE_PATH).to_crs("EPSG:4326")
gate_geom_bounds = gate.total_bounds  # (minx, miny, maxx, maxy)

lat_min, lat_max, lon_min_360, lon_max_360 = gate_bounds(GATE_PATH, lat_buffer=2.0, lon_buffer=5.0)
print(f"📍 Gate: {strait_name}")
print(f"   Bounds: lon [{gate_geom_bounds[0]:.2f}, {gate_geom_bounds[2]:.2f}], "
      f"lat [{gate_geom_bounds[1]:.2f}, {gate_geom_bounds[3]:.2f}]")
print(f"   Filter: lat [{lat_min:.1f}, {lat_max:.1f}], lon [{lon_min_360:.1f}, {lon_max_360:.1f}]")

# ── 3b. Load satellite cycles with spatial + pass filter ──
print(f"\n🔄 Loading {satellite_name} cycles for Pass {PASS_NUMBER}...")
ds_pass = load_cycles_serial(
    base_dir=str(BASE_DIR),
    cycles=cycles,
    pass_number=PASS_NUMBER,
    lat_bounds=(lat_min, lat_max),
    lon_bounds=(lon_min_360, lon_max_360),
    use_flag=USE_FLAG,
    satellite=satellite_name,
)
if ds_pass is None or ds_pass.sizes.get("time", 0) == 0:
    raise ValueError(f"No data found for Pass {PASS_NUMBER}")
print(f"   Raw observations: {ds_pass.sizes['time']:,}")

# ── 3c. Interpolate geoid at observation locations ──
geoid_at_obs = interpolate_geoid(
    geoid_path=GEOID_PATH,
    target_lats=ds_pass["latitude"].values,
    target_lons=ds_pass["longitude"].values,
)
print(f"   Geoid interpolated: {np.sum(np.isfinite(geoid_at_obs)):,}/{len(geoid_at_obs):,} valid")

# ── 3d. Build DOT DataFrame ──
j2_data = build_dataframe(ds_pass, geoid_at_obs, PASS_NUMBER)
if j2_data is None or j2_data.empty:
    raise ValueError(f"No valid data after DOT computation for Pass {PASS_NUMBER}")

n_cycles_loaded = j2_data["cycle"].nunique()
mean_lat = j2_data["lat"].mean()

print(f"\n✅ Data ready!")
print(f"   Observations: {len(j2_data):,}")
print(f"   Cycles: {n_cycles_loaded}")
print(f"   LON: [{j2_data['lon'].min():.3f}, {j2_data['lon'].max():.3f}]")
print(f"   LAT: [{j2_data['lat'].min():.3f}, {j2_data['lat'].max():.3f}]  (mean: {mean_lat:.3f}°)")
print(f"   Time: {j2_data['time'].min()} → {j2_data['time'].max()}")

📍 Gate: Davis Strait
   Bounds: lon [-64.40, -52.98], lat [67.68, 68.16]
   Filter: lat [-1447118.2, -1060463.3], lon [343.5, 171.2]

🔄 Loading J2 cycles for Pass 248...


/Users/nicolocaron/Documents/GitHub/ARCFRESH-DTU-NICO-and-AMALIE/src/slcci/loader.py:23: RuntimeWarning: invalid value encountered in remainder
  return lon % 360
/Users/nicolocaron/Documents/GitHub/ARCFRESH-DTU-NICO-and-AMALIE/src/slcci/loader.py:23: RuntimeWarning: invalid value encountered in remainder
  return lon % 360


ValueError: No data found for Pass 248

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📈 CELL 4: SLOPE TIMELINE  (Tab 1)
# ════════════════════════════════════════════════════════════════
# Build DOT matrix → compute slope per month → plot timeline

dot_matrix, time_periods, lon_centers, x_km = build_dot_matrix(j2_data, lon_bin_size=LON_BIN_SIZE)
slope_series = compute_slope_series(dot_matrix, x_km)
time_array = np.array([pd.Timestamp(str(p)) for p in time_periods])

print(f"DOT matrix: {dot_matrix.shape[0]} lon bins × {dot_matrix.shape[1]} months")
print(f"Valid cells: {np.sum(np.isfinite(dot_matrix)):,} / {dot_matrix.size:,} "
      f"({100*np.sum(np.isfinite(dot_matrix))/dot_matrix.size:.1f}%)")
print(f"Slope range: [{np.nanmin(slope_series):.4f}, {np.nanmax(slope_series):.4f}] m/100km")

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(time_array, slope_series, "-o", markersize=3, linewidth=1.2, color="steelblue")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylabel("DOT Slope (m / 100 km)", fontsize=12)
ax.set_title(f"📈 Slope Timeline — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
             f"{time_array[0].strftime('%Y-%m')} to {time_array[-1].strftime('%Y-%m')}",
             fontsize=14, fontweight="bold")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")

# Add mean ± std annotation
mean_s = np.nanmean(slope_series)
std_s = np.nanstd(slope_series)
ax.axhline(mean_s, color="red", linewidth=1, linestyle="--", alpha=0.7, label=f"Mean: {mean_s:.4f}")
ax.fill_between(time_array, mean_s - std_s, mean_s + std_s, color="red", alpha=0.08, label=f"±1σ: {std_s:.4f}")
ax.legend(fontsize=10)

plt.tight_layout()
if SAVE_OUTPUTS:
    fname = f"01_slope_timeline_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Slope timeline complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📊 CELL 5: DOT PROFILE  (Tab 2)
# ════════════════════════════════════════════════════════════════
# Mean DOT profile (pooled across all times) with coarser binning

profile_mean, lon_centers_p, x_km_p = mean_profile_pooled(j2_data, lon_bin_size=PROFILE_BIN)
valid_p = np.isfinite(profile_mean)

print(f"Profile: {np.sum(valid_p)}/{len(profile_mean)} bins with data ({PROFILE_BIN}° resolution)")

# Linear fit on profile
slope_prof_100km = np.nan
if np.sum(valid_p) >= 2:
    slope_prof, intercept_prof = np.polyfit(x_km_p[valid_p], profile_mean[valid_p], 1)
    fit_line = slope_prof * x_km_p + intercept_prof
    slope_prof_100km = slope_prof * 100.0
    print(f"Profile slope: {slope_prof_100km:.4f} m/100km")

fig, ax = plt.subplots(figsize=(14, 5))
if np.any(valid_p):
    ax.plot(x_km_p[valid_p], profile_mean[valid_p], "o-", color="darkblue",
            markersize=5, linewidth=2, label="Mean DOT")
    if np.sum(valid_p) >= 2:
        ax.plot(x_km_p, fit_line, "--", color="red", linewidth=1.5,
                label=f"Linear fit: {slope_prof_100km:.4f} m/100km")

ax.set_xlabel("Distance along gate (km)", fontsize=12)
ax.set_ylabel("DOT (m)", fontsize=12)
ax.set_title(f"📊 Mean DOT Profile — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
             f"Pooled across {len(time_periods)} months, {PROFILE_BIN}° bins",
             fontsize=14, fontweight="bold")

# WEST / EAST labels
if np.any(valid_p):
    y_range = np.nanmax(profile_mean[valid_p]) - np.nanmin(profile_mean[valid_p])
    y_top = np.nanmax(profile_mean[valid_p]) - 0.05 * y_range if y_range > 0 else np.nanmean(profile_mean[valid_p])
    ax.text(x_km_p.min(), y_top, "WEST", fontsize=11, fontweight="bold", ha="left", va="top")
    ax.text(x_km_p.max(), y_top, "EAST", fontsize=11, fontweight="bold", ha="right", va="top")

ax.legend(fontsize=10)
plt.tight_layout()
if SAVE_OUTPUTS:
    fname = f"02_dot_profile_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ DOT profile complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🗺️ CELL 6: SPATIAL MAP  (Tab 3)
# ════════════════════════════════════════════════════════════════

# Compute mean DOT per (lon, lat) location
mean_dot_by_loc = j2_data.groupby(["lon", "lat"]).agg({"dot": "mean"}).reset_index()

proj = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(14, 10), subplot_kw=dict(projection=proj))

# Map extent
LON_BUF, LAT_BUF = 3.0, 2.0
ax.set_extent([
    gate_geom_bounds[0] - LON_BUF, gate_geom_bounds[2] + LON_BUF,
    gate_geom_bounds[1] - LAT_BUF, gate_geom_bounds[3] + LAT_BUF,
], crs=proj)

ax.coastlines(linewidth=1.0)
ax.add_feature(cfeature.LAND, facecolor="beige", alpha=0.7)
ax.add_feature(cfeature.OCEAN, facecolor="lightblue", alpha=0.3)
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                   linewidth=0.8, alpha=0.6, linestyle="--", color="gray")
gl.top_labels = False
gl.right_labels = False

# Gate line
gate.plot(ax=ax, color="red", linewidth=3, transform=proj, zorder=10, label="Gate")

# DOT scatter
sc = ax.scatter(
    mean_dot_by_loc["lon"], mean_dot_by_loc["lat"],
    c=mean_dot_by_loc["dot"], s=15, alpha=0.7,
    cmap="viridis", transform=proj, zorder=5,
    edgecolors="white", linewidths=0.2,
)
plt.colorbar(sc, ax=ax, label="DOT (m)", shrink=0.8, pad=0.02)
ax.legend(loc="upper left", fontsize=12, framealpha=0.95)

ax.set_title(
    f"🗺️ Mean DOT Map — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
    f"{j2_data['time'].min().strftime('%Y-%m-%d')} to {j2_data['time'].max().strftime('%Y-%m-%d')} "
    f"({len(j2_data):,} obs)",
    fontsize=14, fontweight="bold", pad=15,
)

plt.tight_layout()
if SAVE_OUTPUTS:
    fname = f"03_spatial_map_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Spatial map complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📅 CELL 7: MONTHLY ANALYSIS — 12 subplots  (Tab 4)
# ════════════════════════════════════════════════════════════════

monthly_profiles, lon_c_monthly, x_km_monthly = monthly_climatology_profiles(
    j2_data, lon_bin_size=PROFILE_BIN
)

MONTH_NAMES = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December",
}

# Global DOT range for consistent y-limits
all_vals = np.concatenate([p[np.isfinite(p)] for p in monthly_profiles.values() if np.any(np.isfinite(p))])
if len(all_vals) > 0:
    dot_margin = 0.05 * (all_vals.max() - all_vals.min())
    ylim_global = (all_vals.min() - dot_margin, all_vals.max() + dot_margin)
else:
    ylim_global = None

fig, axes = plt.subplots(3, 4, figsize=(28, 18))
axes = axes.flatten()
monthly_results = []

for month_num in range(1, 13):
    ax = axes[month_num - 1]
    prof = monthly_profiles[month_num]
    valid = np.isfinite(prof)

    if not np.any(valid):
        ax.text(0.5, 0.5, f"{MONTH_NAMES[month_num]}\n\nNo data",
                ha="center", va="center", fontsize=13, transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
        continue

    # Scatter binned DOT
    ax.scatter(x_km_monthly[valid], prof[valid], s=40, alpha=0.75,
               color="steelblue", edgecolor="k", linewidth=0.3)

    # Linear fit
    if np.sum(valid) >= 2:
        slope_m, intercept_m = np.polyfit(x_km_monthly[valid], prof[valid], 1)
        fit_y = slope_m * x_km_monthly[valid] + intercept_m
        ax.plot(x_km_monthly[valid], fit_y, "--", linewidth=2, color="darkred")

        ss_res = np.sum((prof[valid] - fit_y) ** 2)
        ss_tot = np.sum((prof[valid] - np.mean(prof[valid])) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        slope_100km = slope_m * 100.0

        monthly_results.append({
            "month": month_num,
            "month_name": MONTH_NAMES[month_num],
            "slope_m_100km": slope_100km,
            "r_squared": r2,
            "n_bins": int(np.sum(valid)),
        })
        ax.set_title(f"{MONTH_NAMES[month_num]}\nslope={slope_100km:.4f} m/100km | R²={r2:.3f}",
                     fontsize=11, fontweight="bold")
    else:
        ax.set_title(f"{MONTH_NAMES[month_num]}", fontsize=11, fontweight="bold")

    ax.set_xlabel("Distance (km)", fontsize=10)
    ax.set_ylabel("DOT (m)", fontsize=10)
    if ylim_global:
        ax.set_ylim(ylim_global)

fig.suptitle(
    f"📅 Monthly DOT Climatology — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
    f"Binned at {PROFILE_BIN}° resolution with linear regression",
    fontsize=17, fontweight="bold", y=0.995,
)
plt.tight_layout(rect=[0, 0, 1, 0.97])
if SAVE_OUTPUTS:
    fname = f"04_monthly_analysis_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()

# Summary table
print("\n" + "=" * 70)
print("📊 MONTHLY SUMMARY")
print("=" * 70)
for r in monthly_results:
    print(f"{r['month_name']:12s}: slope={r['slope_m_100km']:+.4f} m/100km, "
          f"R²={r['r_squared']:.3f}, {r['n_bins']} bins")
print("=" * 70)
print("✅ Monthly analysis complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🌀 CELL 8: GEOSTROPHIC VELOCITY  (Tab 5)
# ════════════════════════════════════════════════════════════════
# v_geo = -(g/f) × ∂η/∂x   [SLCCI DOT slope method]

v_geo = geostrophic_velocity_from_slope(slope_series, mean_lat)

f_param = coriolis_parameter(mean_lat)
print(f"Mean latitude: {mean_lat:.3f}°")
print(f"Coriolis parameter: f = {f_param:.6e} rad/s")
print(f"v_geo range: [{np.nanmin(v_geo):.4f}, {np.nanmax(v_geo):.4f}] m/s")
print(f"v_geo mean:  {np.nanmean(v_geo):.4f} ± {np.nanstd(v_geo):.4f} m/s")

# ── Plot: v_geo timeline ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# Panel 1: slope
ax1.plot(time_array, slope_series, "-o", markersize=3, linewidth=1.2, color="steelblue")
ax1.axhline(0, color="k", linewidth=0.8)
ax1.set_ylabel("DOT Slope (m / 100 km)", fontsize=12)
ax1.set_title(f"🌀 DOT Slope → Geostrophic Velocity — {strait_name} — {satellite_name} Pass {PASS_NUMBER}",
              fontsize=14, fontweight="bold")

# Panel 2: v_geo
ax2.plot(time_array, v_geo, "-o", markersize=3, linewidth=1.2, color="darkgreen")
ax2.axhline(0, color="k", linewidth=0.8)
ax2.set_ylabel("v_geo (m/s)", fontsize=12)
ax2.set_xlabel("Date", fontsize=12)

# Mean line
mean_v = np.nanmean(v_geo)
ax2.axhline(mean_v, color="red", linestyle="--", linewidth=1, alpha=0.7,
            label=f"Mean: {mean_v:.4f} m/s")
ax2.legend(fontsize=10)

ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"05_geostrophic_velocity_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Geostrophic velocity complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🏔️ CELL 9: LOAD BATHYMETRY (GEBCO) + GATE GEOMETRY
# ════════════════════════════════════════════════════════════════

print("🔄 Loading GEBCO bathymetry...")
ds_gebco = xr.open_dataset(GEBCO_PATH)

# Get gate profile points
gate_lon, gate_lat, gate_x_km = gate_profile_points(GATE_PATH, n_points=100)
print(f"   Gate sampled: {len(gate_lon)} points")
print(f"   Gate length: {gate_x_km[-1]:.1f} km")

# Interpolate GEBCO elevation at gate points
gebco_lat = ds_gebco["lat"].values
gebco_lon = ds_gebco["lon"].values
gebco_elev = ds_gebco["elevation"].values

interp_gebco = RegularGridInterpolator(
    (gebco_lat, gebco_lon), gebco_elev.astype(float),
    method="linear", bounds_error=False, fill_value=np.nan
)

# Wrap gate longitudes to [0, 360) for GEBCO
gate_lon_360 = gate_lon % 360
gate_elevation = interp_gebco(np.column_stack([gate_lat, gate_lon_360]))
gate_depth = np.abs(gate_elevation)  # depth positive down
gate_depth = np.where(gate_elevation > 0, 0, gate_depth)  # land = 0 depth

# Cap depth
gate_depth_capped = np.minimum(gate_depth, DEPTH_CAP_M)

# Compute dx (segment width) in meters
dx_m = np.gradient(gate_x_km * 1000.0)  # km → m

ds_gebco.close()

print(f"   Depth range: {gate_depth.min():.0f} – {gate_depth.max():.0f} m")
print(f"   Depth capped at {DEPTH_CAP_M:.0f} m")
print(f"   dx range: {dx_m.min():.0f} – {dx_m.max():.0f} m")

# ── Plot: bathymetry profile ──
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(gate_x_km, 0, -gate_depth, color="saddlebrown", alpha=0.5, label="Bathymetry")
ax.axhline(-DEPTH_CAP_M, color="red", linestyle="--", linewidth=1.5, label=f"Depth cap ({DEPTH_CAP_M:.0f} m)")
ax.set_xlabel("Distance along gate (km)", fontsize=12)
ax.set_ylabel("Elevation (m)", fontsize=12)
ax.set_title(f"��️ GEBCO Bathymetry along {strait_name}", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(-min(gate_depth.max() * 1.1, 1000), 50)
plt.tight_layout()
if SAVE_OUTPUTS:
    fname = f"06_bathymetry_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Bathymetry loaded")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🌊 CELL 10: VOLUME TRANSPORT  (Tab 6)
# ════════════════════════════════════════════════════════════════
# VT(t) = Σᵢ v_perp(i,t) · H(i) · dx(i)  /  10⁶  [Sv]
#
# For SLCCI: v_perp is uniform across the gate (single slope → single v_geo)

n_gate_pts = len(gate_lon)
n_time = len(v_geo)

# Build (N_pts, N_time) velocity array — v_geo is uniform along gate
v_perp_matrix = np.tile(v_geo, (n_gate_pts, 1))  # (N_pts, N_time)

# Volume transport via src/physics/transport
vt_sv = volume_transport(
    v_perp=v_perp_matrix,
    depth=gate_depth,
    dx=dx_m,
    depth_cap=DEPTH_CAP_M,
)

print(f"Volume Transport:")
print(f"   Range: [{np.nanmin(vt_sv):.3f}, {np.nanmax(vt_sv):.3f}] Sv")
print(f"   Mean:  {np.nanmean(vt_sv):.3f} ± {np.nanstd(vt_sv):.3f} Sv")

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(time_array, vt_sv, "-o", markersize=3, linewidth=1.2, color="teal")
ax.axhline(0, color="k", linewidth=0.8)
mean_vt = np.nanmean(vt_sv)
ax.axhline(mean_vt, color="red", linestyle="--", linewidth=1, alpha=0.7,
           label=f"Mean: {mean_vt:.3f} Sv")
ax.fill_between(time_array, 0, vt_sv,
                where=vt_sv >= 0, color="teal", alpha=0.15, interpolate=True)
ax.fill_between(time_array, 0, vt_sv,
                where=vt_sv < 0, color="orange", alpha=0.15, interpolate=True)

ax.set_ylabel("Volume Transport (Sv)", fontsize=12)
ax.set_title(
    f"🌊 Volume Transport — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
    f"Depth cap: {DEPTH_CAP_M:.0f} m | Positive = into Arctic",
    fontsize=14, fontweight="bold",
)
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"07_volume_transport_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Volume transport complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🧂 CELL 11: SALINITY PROFILE  (Tab 8)
# ════════════════════════════════════════════════════════════════

psal_file = PSAL_DIR / STRAIT_PSAL_MAP[SELECTED_STRAIT]
print(f"🔄 Loading salinity: {psal_file.name}")

ds_psal = xr.open_dataset(psal_file)
psal_lat = ds_psal["latitude"].values        # (nb_prof,)
psal_lon = ds_psal["longitude"].values        # (nb_prof,)
psal_depth = ds_psal["depth"].values           # (z,)
psal_data = ds_psal["PSAL"].values             # (time=12, z, nb_prof)

print(f"   Profiles: {len(psal_lat)} points")
print(f"   Depth levels: {len(psal_depth)} ({psal_depth.min():.0f} – {psal_depth.max():.0f} m)")
print(f"   PSAL range: [{np.nanmin(psal_data):.2f}, {np.nanmax(psal_data):.2f}] PSU")

# ── Plot 1: depth profiles + monthly SSS ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Left: annual mean salinity vs depth
annual_mean_psal = np.nanmean(psal_data, axis=0)  # (z, nb_prof)
for ip in range(annual_mean_psal.shape[1]):
    profile = annual_mean_psal[:, ip]
    valid_z = np.isfinite(profile)
    if np.any(valid_z):
        ax1.plot(profile[valid_z], psal_depth[valid_z], alpha=0.4, linewidth=0.8, color="steelblue")

overall_profile = np.nanmean(annual_mean_psal, axis=1)  # (z,)
valid_overall = np.isfinite(overall_profile)
ax1.plot(overall_profile[valid_overall], psal_depth[valid_overall],
         linewidth=3, color="darkblue", label="Mean profile")

ax1.axhline(DEPTH_CAP_M, color="red", linestyle="--", linewidth=1.5,
            label=f"Depth cap ({DEPTH_CAP_M:.0f} m)")
ax1.axvline(S_REF_PSU, color="orange", linestyle=":", linewidth=1.5,
            label=f"S_ref = {S_REF_PSU} PSU")
ax1.invert_yaxis()
ax1.set_xlabel("Salinity (PSU)", fontsize=12)
ax1.set_ylabel("Depth (m)", fontsize=12)
ax1.set_title("Annual Mean Salinity Profiles", fontsize=13, fontweight="bold")
ax1.legend(fontsize=10)
ax1.set_ylim(min(psal_depth.max(), 1000), 0)

# Right: monthly SSS along gate
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
colors = plt.cm.twilight_shifted(np.linspace(0, 1, 12))
for im in range(12):
    sss_month = psal_data[im, 0, :]
    ax2.plot(psal_lon, sss_month, "o-", markersize=4, linewidth=1.2,
             color=colors[im], label=month_labels[im], alpha=0.8)

ax2.set_xlabel("Longitude (°)", fontsize=12)
ax2.set_ylabel("SSS (PSU)", fontsize=12)
ax2.set_title("Monthly Surface Salinity along Gate", fontsize=13, fontweight="bold")
ax2.legend(fontsize=8, ncol=4, loc="best")

fig.suptitle(f"🧂 Salinity Profile — {strait_name} (ISAS25 PSAL Climatology)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"08_salinity_profile_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
ds_psal.close()
print("✅ Salinity profile complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 💧 CELL 12: FRESHWATER TRANSPORT  (Tab 7)
# ════════════════════════════════════════════════════════════════
# FW(t) = Σᵢ v_perp(i,t) · (1 − S(i,t)/S_ref) · H(i) · dx(i)

print("🔄 Computing freshwater transport...")

# ── Map ISAS25 SSS to gate points ──
ds_psal = xr.open_dataset(PSAL_DIR / STRAIT_PSAL_MAP[SELECTED_STRAIT])
psal_lon_isas = ds_psal["longitude"].values   # (nb_prof,)
psal_sss_monthly = ds_psal["PSAL"].values[:, 0, :]  # (12, nb_prof) — surface layer

# Interpolate SSS from ISAS25 profiles to gate points (nearest-neighbor)
sss_at_gate = np.full((12, n_gate_pts), np.nan)
for im in range(12):
    sss_month = psal_sss_monthly[im, :]
    valid_isas = np.isfinite(sss_month)
    if np.sum(valid_isas) < 2:
        continue
    for ig in range(n_gate_pts):
        dist = np.abs(psal_lon_isas[valid_isas] - gate_lon[ig])
        idx_nearest = np.argmin(dist)
        sss_at_gate[im, ig] = sss_month[valid_isas][idx_nearest]

print(f"   SSS at gate: {np.sum(np.isfinite(sss_at_gate)):,}/{sss_at_gate.size} valid")

# ── Build SSS matrix (N_pts, N_time) — cycle through 12 months ──
time_months = np.array([pd.Timestamp(str(p)).month for p in time_periods])
sss_matrix = np.full((n_gate_pts, n_time), np.nan)
for it in range(n_time):
    month_idx = time_months[it] - 1  # 0-based
    sss_matrix[:, it] = sss_at_gate[month_idx, :]

print(f"   SSS matrix: {sss_matrix.shape}, "
      f"{np.sum(np.isfinite(sss_matrix)):,}/{sss_matrix.size} valid")

# ── Freshwater transport ──
fw_m3s = freshwater_transport(
    v_perp=v_perp_matrix,
    sss=sss_matrix,
    depth=gate_depth,
    dx=dx_m,
    depth_cap=DEPTH_CAP_M,
    s_ref=S_REF_PSU,
)
fw_msv = fw_m3s / 1e3  # m³/s → mSv

print(f"\nFreshwater Transport:")
print(f"   Range: [{np.nanmin(fw_m3s):.0f}, {np.nanmax(fw_m3s):.0f}] m³/s")
print(f"   Mean:  {np.nanmean(fw_msv):.1f} ± {np.nanstd(fw_msv):.1f} mSv")

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(time_array, fw_msv, "-o", markersize=3, linewidth=1.2, color="royalblue")
ax.axhline(0, color="k", linewidth=0.8)
mean_fw = np.nanmean(fw_msv)
ax.axhline(mean_fw, color="red", linestyle="--", linewidth=1, alpha=0.7,
           label=f"Mean: {mean_fw:.1f} mSv")
ax.fill_between(time_array, 0, fw_msv,
                where=fw_msv >= 0, color="royalblue", alpha=0.12, interpolate=True)
ax.fill_between(time_array, 0, fw_msv,
                where=fw_msv < 0, color="orange", alpha=0.12, interpolate=True)

ax.set_ylabel("Freshwater Transport (mSv)", fontsize=12)
ax.set_title(
    f"💧 Freshwater Transport — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
    f"S_ref = {S_REF_PSU} PSU | Depth cap: {DEPTH_CAP_M:.0f} m | Positive = into Arctic",
    fontsize=14, fontweight="bold",
)
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"09_freshwater_transport_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
ds_psal.close()
print("✅ Freshwater transport complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🧪 CELL 13: SALT FLUX  (Tab 9)
# ════════════════════════════════════════════════════════════════
# SF(t) = Σᵢ ρ · (SSS(i,t)/1000) · v_perp(i,t) · H(i) · dx(i)  [kg/s]

sf_kgs = salt_flux(
    v_perp=v_perp_matrix,
    sss=sss_matrix,
    depth=gate_depth,
    dx=dx_m,
    depth_cap=DEPTH_CAP_M,
    rho=RHO,
)
sf_kts = sf_kgs / 1e6  # kg/s → kt/s

print(f"Salt Flux:")
print(f"   Range: [{np.nanmin(sf_kgs):.0f}, {np.nanmax(sf_kgs):.0f}] kg/s")
print(f"   Mean:  {np.nanmean(sf_kts):.2f} ± {np.nanstd(sf_kts):.2f} kt/s")

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(time_array, sf_kts, "-o", markersize=3, linewidth=1.2, color="darkorange")
ax.axhline(0, color="k", linewidth=0.8)
mean_sf = np.nanmean(sf_kts)
ax.axhline(mean_sf, color="red", linestyle="--", linewidth=1, alpha=0.7,
           label=f"Mean: {mean_sf:.2f} kt/s")
ax.fill_between(time_array, 0, sf_kts,
                where=sf_kts >= 0, color="darkorange", alpha=0.15, interpolate=True)
ax.fill_between(time_array, 0, sf_kts,
                where=sf_kts < 0, color="purple", alpha=0.15, interpolate=True)

ax.set_ylabel("Salt Flux (kt/s)", fontsize=12)
ax.set_title(
    f"🧪 Salt Flux — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
    f"ρ = {RHO} kg/m³ | Depth cap: {DEPTH_CAP_M:.0f} m | Positive = into Arctic",
    fontsize=14, fontweight="bold",
)
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"10_salt_flux_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Salt flux complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📊 CELL 14: TEMPORAL AGGREGATION — Monthly, Climatology, Annual
# ════════════════════════════════════════════════════════════════

time_idx = pd.DatetimeIndex(time_array)

# ── Monthly means ──
vt_monthly = monthly_mean(vt_sv, time_idx)
fw_monthly = monthly_mean(fw_msv, time_idx)
vgeo_monthly = monthly_mean(v_geo, time_idx)

# ── Climatology (all years collapsed) ──
vt_clim = monthly_climatology(vt_sv, time_idx)
fw_clim = monthly_climatology(fw_msv, time_idx)
vgeo_clim = monthly_climatology(v_geo, time_idx)

# ── Annual means ──
vt_annual = annual_mean(vt_sv, time_idx)
fw_annual = annual_mean(fw_msv, time_idx)

# ── Rolling mean (12-month window) ──
vt_rolling = rolling_mean(vt_sv, window=12)

# ── Plot: 2×2 aggregation overview ──
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# (0,0) Monthly climatology: VT
ax = axes[0, 0]
months = vt_clim["month"].values
ax.bar(months, vt_clim["mean"], yerr=vt_clim["std"], color="teal", alpha=0.7,
       capsize=4, edgecolor="k", linewidth=0.5)
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_ylabel("VT (Sv)", fontsize=12)
ax.set_title("Monthly Climatology — Volume Transport", fontsize=13, fontweight="bold")

# (0,1) Monthly climatology: FW
ax = axes[0, 1]
ax.bar(fw_clim["month"].values, fw_clim["mean"], yerr=fw_clim["std"],
       color="royalblue", alpha=0.7, capsize=4, edgecolor="k", linewidth=0.5)
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_ylabel("FW (mSv)", fontsize=12)
ax.set_title("Monthly Climatology — Freshwater Transport", fontsize=13, fontweight="bold")

# (1,0) Annual mean: VT
ax = axes[1, 0]
ax.bar(vt_annual.index.year, vt_annual["value"], color="teal", alpha=0.7,
       edgecolor="k", linewidth=0.5)
ax.axhline(np.nanmean(vt_sv), color="red", linestyle="--", linewidth=1,
           label=f"Overall mean: {np.nanmean(vt_sv):.3f} Sv")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylabel("VT (Sv)", fontsize=12)
ax.set_xlabel("Year", fontsize=12)
ax.set_title("Annual Mean — Volume Transport", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

# (1,1) Time series with rolling mean
ax = axes[1, 1]
ax.plot(time_array, vt_sv, alpha=0.3, color="teal", linewidth=0.8, label="Monthly")
ax.plot(time_array, vt_rolling, color="darkred", linewidth=2.5, label="12-month rolling")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylabel("VT (Sv)", fontsize=12)
ax.set_xlabel("Date", fontsize=12)
ax.set_title("Volume Transport with Rolling Mean", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

fig.suptitle(
    f"📊 Temporal Aggregation — {strait_name} — {satellite_name} Pass {PASS_NUMBER}",
    fontsize=16, fontweight="bold", y=1.01,
)
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"11_temporal_aggregation_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Temporal aggregation complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📤 CELL 15: EXPORT SUMMARY TABLE
# ════════════════════════════════════════════════════════════════

summary_df = pd.DataFrame({
    "date": time_array,
    "slope_m_100km": slope_series,
    "v_geo_m_s": v_geo,
    "VT_Sv": vt_sv,
    "FW_m3_s": fw_m3s,
    "FW_mSv": fw_msv,
    "SF_kg_s": sf_kgs,
    "SF_kt_s": sf_kts,
})

print("\n" + "=" * 80)
print(f"📤 EXPORT SUMMARY — {strait_name} — {satellite_name} Pass {PASS_NUMBER}")
print("=" * 80)
print(f"\n{'Variable':<25s} {'Mean':>12s} {'Std':>12s} {'Min':>12s} {'Max':>12s}")
print("-" * 73)
for col in ["slope_m_100km", "v_geo_m_s", "VT_Sv", "FW_mSv", "SF_kt_s"]:
    vals = summary_df[col].dropna()
    print(f"{col:<25s} {vals.mean():>12.4f} {vals.std():>12.4f} {vals.min():>12.4f} {vals.max():>12.4f}")
print("-" * 73)

# Save to CSV
if SAVE_OUTPUTS:
    csv_path = OUTPUT_DIR / f"summary_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.csv"
    summary_df.to_csv(csv_path, index=False)
    print(f"\n💾 CSV saved: {csv_path}")

    # Save to Excel
    xlsx_path = OUTPUT_DIR / f"summary_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.xlsx"
    try:
        with pd.ExcelWriter(xlsx_path) as writer:
            summary_df.to_excel(writer, sheet_name="TimeSeries", index=False)
            vt_clim.to_excel(writer, sheet_name="VT_Climatology", index=False)
            fw_clim.to_excel(writer, sheet_name="FW_Climatology", index=False)
            vgeo_clim.to_excel(writer, sheet_name="Vgeo_Climatology", index=False)
        print(f"💾 Excel saved: {xlsx_path}")
    except ImportError:
        print("⚠️ openpyxl not available — skipping Excel export")

print("\n" + "=" * 80)
print("✅ ALL ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\n📊 Outputs saved to: {OUTPUT_DIR}")
print(f"   11 plots + 1 CSV + 1 Excel")

---

## 📋 Analysis Summary

This notebook covers the **complete SLCCI analysis pipeline**:

| Step | What | Module |
|------|------|--------|
| Load | Cycle files → geoid → DOT DataFrame | `src.slcci.loader`, `geoid`, `dot` |
| Slope | DOT matrix → linear fit → m/100km | `src.slcci.dot.compute_slope_series` |
| Profile | Pooled mean DOT vs distance | `src.slcci.binning.mean_profile_pooled` |
| Map | Cartopy spatial visualization | matplotlib + cartopy |
| Monthly | 12-month climatological profiles | `src.slcci.binning.monthly_climatology_profiles` |
| v_geo | v = −(g/f) × ∂η/∂x | `src.physics.geostrophy.geostrophic_velocity_from_slope` |
| VT | Σ v·H·dx / 10⁶ [Sv] | `src.physics.transport.volume_transport` |
| FW | Σ v·(1−S/S_ref)·H·dx [m³/s] | `src.physics.transport.freshwater_transport` |
| Salt | Σ ρ·(S/1000)·v·H·dx [kg/s] | `src.physics.transport.salt_flux` |
| Aggregation | Monthly/annual/rolling means | `src.physics.aggregation` |

**Key assumptions**:
- SLCCI provides a **single DOT slope** per time step → uniform v_geo across the gate
- Salinity from **ISAS25 climatology** (12-month cycle, repeated each year)
- Bathymetry from **GEBCO 2025**, capped at depth_cap
- Positive transport = **into the Arctic**